# Ablation: Hybrid Combination Methods

Tests different strategies for combining sparse (BM25, TF-IDF) and dense retrieval:
1. **Weighted score fusion** — vary TF-IDF vs Dense weights
2. **BM25 inclusion** — add BM25 scores to the fusion
3. **Reciprocal Rank Fusion (RRF)** — rank-based combination, vary k
4. **Intersection** — only return passages in all three candidate sets

Base retrievers are fit once and shared across all fusion strategies.


In [1]:
import sys, os, gc
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

from loader import load_data
from retrievers.bm25 import BM25Retriever
from retrievers.tf_idf_retriever import TFIDFRetriever
from retrievers.dense_retriever import DenseRetriever
from evaluation.mrr import mrr_at_10
from evaluation.timing import measure_retrieval_time

N = 50_000
TOP_K = 10
CANDIDATE_K = 50

ds = load_data(n=N)
passages_text, queries = [], []
for example in ds:
    queries.append(example["query"])
    for p in example["passages"]["passage_text"]:
        passages_text.append(p)
print(f"Passages: {len(passages_text)}, Queries: {len(queries)}")

/home/skimura/Projects/ics624_document_retrieval_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Passages: 498725, Queries: 50000


In [2]:
print("Fitting base retrievers...")
bm25  = BM25Retriever(top_k=CANDIDATE_K)
tfidf = TFIDFRetriever(top_k=CANDIDATE_K)
dense = DenseRetriever(top_k=CANDIDATE_K)

bm25.fit(passages_text)
tfidf.fit(passages_text)
dense.fit("sbert_embeddings.npy", passages_text)
print("Done.")

Fitting base retrievers...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13154.69it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SentenceBERTEncoder using device: cuda
Done.


## 1. Weighted Score Fusion (TF-IDF + Dense)

Sweeps the dense weight from 0 (TF-IDF only) to 1 (Dense only).

In [3]:
def make_weighted_retriever(w_dense, w_tfidf, w_bm25=0.0, top_k=TOP_K):
    class _Retriever:
        def query(self, query):
            bm25_c  = bm25.query(query)
            tfidf_c = tfidf.query(query)
            q_emb = dense.encoder.encode([query]).astype(np.float32)
            q_emb /= np.linalg.norm(q_emb, axis=1, keepdims=True)
            _, dt = dense.index.search(q_emb, CANDIDATE_K)
            dense_c = dt[0].tolist()
            union = list(set(bm25_c) | set(tfidf_c) | set(dense_c))
            cand = np.array(union)
            d_s = (dense.embeddings[cand] @ q_emb.T).ravel()
            d_s = np.clip(d_s, 0, None) / max(d_s.max(), 1e-9)
            t_s = tfidf.score_candidates(query, union)
            b_s = bm25.score_candidates(query, union) if w_bm25 > 0 else np.zeros(len(union))
            combined = w_dense * d_s + w_tfidf * t_s + w_bm25 * b_s
            order = np.argsort(combined)[::-1]
            return [union[i] for i in order[:top_k]]

        def query_batch(self, qs):
            results = []
            tfidf_batch = tfidf.query_batch(qs)
            bm25_batch  = bm25.query_batch(qs)
            q_embs = dense.encoder.encode(qs).astype(np.float32)
            q_embs /= np.linalg.norm(q_embs, axis=1, keepdims=True)
            _, dense_batch = dense.index.search(q_embs, CANDIDATE_K)
            tq = tfidf.vectorizer.transform(qs)
            bq = bm25.vectorizer.transform(qs) if w_bm25 > 0 else None
            for i, q in enumerate(qs):
                union = list(set(bm25_batch[i]) | set(tfidf_batch[i]) | set(dense_batch[i].tolist()))
                cand = np.array(union)
                d_s = (dense.embeddings[cand] @ q_embs[i]).ravel()
                d_s = np.clip(d_s, 0, None) / max(d_s.max(), 1e-9)
                t_s = cosine_similarity(tq[i], tfidf.tf_idf_matrix[cand])[0]
                t_s = t_s / max(t_s.max(), 1e-9)
                b_s = (bq[i] @ bm25.bm25_matrix[cand].T).toarray().ravel() if w_bm25 > 0 else np.zeros(len(union))
                if w_bm25 > 0: b_s = b_s / max(b_s.max(), 1e-9)
                combined = w_dense * d_s + w_tfidf * t_s + w_bm25 * b_s
                order = np.argsort(combined)[::-1]
                results.append([union[j] for j in order[:top_k]])
            return results
    return _Retriever()

In [4]:
results = []

weight_configs = [
    ("Dense only",            1.00, 0.00, 0.00),
    ("Dense 0.9 + TF-IDF 0.1",0.90, 0.10, 0.00),
    ("Dense 0.75 + TF-IDF 0.25",0.75,0.25, 0.00),
    ("Dense 0.5 + TF-IDF 0.5", 0.50, 0.50, 0.00),
    ("Dense 0.25 + TF-IDF 0.75",0.25,0.75, 0.00),
    ("TF-IDF only",            0.00, 1.00, 0.00),
    ("Dense 0.75 + BM25 0.25", 0.75, 0.00, 0.25),
    ("Dense 0.6 + BM25 0.3 + TF-IDF 0.1", 0.60, 0.10, 0.30),
]

for name, wd, wt, wb in weight_configs:
    r = make_weighted_retriever(wd, wt, wb)
    mrr = mrr_at_10(r, ds)
    t   = measure_retrieval_time(r, queries)
    results.append({"Strategy": name, "MRR@10": round(mrr, 4), "Avg ms/query": round(t*1000, 3)})
    print(f"{name:<42} MRR: {mrr:.4f}, {t*1000:.1f} ms/q")

Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100


Batches: 100%|██████████| 1/1 [00:00<00:00, 263.53it/s]


Dense only                                 MRR: 0.5363, 379.8 ms/q
Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100


Batches: 100%|██████████| 1/1 [00:00<00:00, 277.75it/s]


Dense 0.9 + TF-IDF 0.1                     MRR: 0.5587, 377.7 ms/q
Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100


...

TF-IDF only                                MRR: 0.2419, 385.0 ms/q
Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100


Batches: 100%|██████████| 1/1 [00:00<00:00, 331.23it/s]


Dense 0.75 + BM25 0.25                     MRR: 0.5316, 402.2 ms/q
Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100


Batches: 100%|██████████| 1/1 [00:00<00:00, 261.52it/s]

Dense 0.6 + BM25 0.3 + TF-IDF 0.1          MRR: 0.4872, 387.5 ms/q


## 2. Reciprocal Rank Fusion (RRF)

RRF score = Σ 1/(k + rank). Sweeps k ∈ {10, 30, 60, 100}.

In [5]:
def make_rrf_retriever(k=60, top_k=TOP_K):
    class _Retriever:
        def query(self, query):
            bm25_r  = bm25.query(query)
            tfidf_r = tfidf.query(query)
            q_emb = dense.encoder.encode([query]).astype(np.float32)
            q_emb /= np.linalg.norm(q_emb, axis=1, keepdims=True)
            _, dt = dense.index.search(q_emb, CANDIDATE_K)
            dense_r = dt[0].tolist()
            scores = {}
            for ranked in [bm25_r, tfidf_r, dense_r]:
                for rank, idx in enumerate(ranked):
                    scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
            return sorted(scores, key=scores.__getitem__, reverse=True)[:top_k]

        def query_batch(self, qs):
            results = []
            bm25_batch  = bm25.query_batch(qs)
            tfidf_batch = tfidf.query_batch(qs)
            q_embs = dense.encoder.encode(qs).astype(np.float32)
            q_embs /= np.linalg.norm(q_embs, axis=1, keepdims=True)
            _, dense_batch = dense.index.search(q_embs, CANDIDATE_K)
            for i in range(len(qs)):
                scores = {}
                for ranked in [bm25_batch[i], tfidf_batch[i], dense_batch[i].tolist()]:
                    for rank, idx in enumerate(ranked):
                        scores[idx] = scores.get(idx, 0.0) + 1.0 / (k + rank + 1)
                results.append(sorted(scores, key=scores.__getitem__, reverse=True)[:top_k])
            return results
    return _Retriever()

for rrf_k in [10, 30, 60, 100]:
    r = make_rrf_retriever(k=rrf_k)
    mrr = mrr_at_10(r, ds)
    t   = measure_retrieval_time(r, queries)
    results.append({"Strategy": f"RRF (k={rrf_k})", "MRR@10": round(mrr, 4), "Avg ms/query": round(t*1000, 3)})
    print(f"RRF k={rrf_k:<4} MRR: {mrr:.4f}, {t*1000:.1f} ms/q")

Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100


Batches: 100%|██████████| 1/1 [00:00<00:00, 328.35it/s]


RRF k=10   MRR: 0.3994, 385.1 ms/q
Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100


Batches: 100%|██████████| 1/1 [00:00<00:00, 339.04it/s]


RRF k=30   MRR: 0.4007, 385.4 ms/q
Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100


Batches: 100%|██████████| 1/1 [00:00<00:00, 271.56it/s]


RRF k=60   MRR: 0.3986, 388.9 ms/q
Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100


Batches: 100%|██████████| 1/1 [00:00<00:00, 316.77it/s]

RRF k=100  MRR: 0.3975, 387.3 ms/q


## 3. Intersection

Only returns passages that appear in all three candidate sets. May have fewer than top_k results when the intersection is small.

In [6]:
class IntersectionRetriever:
    def query(self, query):
        bm25_s  = set(bm25.query(query))
        tfidf_s = set(tfidf.query(query))
        q_emb = dense.encoder.encode([query]).astype(np.float32)
        q_emb /= np.linalg.norm(q_emb, axis=1, keepdims=True)
        _, dt = dense.index.search(q_emb, CANDIDATE_K)
        dense_s = set(dt[0].tolist())
        return list(bm25_s & tfidf_s & dense_s)[:TOP_K]

    def query_batch(self, qs):
        results = []
        bm25_batch  = bm25.query_batch(qs)
        tfidf_batch = tfidf.query_batch(qs)
        q_embs = dense.encoder.encode(qs).astype(np.float32)
        q_embs /= np.linalg.norm(q_embs, axis=1, keepdims=True)
        _, dense_batch = dense.index.search(q_embs, CANDIDATE_K)
        for i in range(len(qs)):
            inter = set(bm25_batch[i]) & set(tfidf_batch[i]) & set(dense_batch[i].tolist())
            results.append(list(inter)[:TOP_K])
        return results

r = IntersectionRetriever()
mrr = mrr_at_10(r, ds)
t   = measure_retrieval_time(r, queries)
results.append({"Strategy": "Intersection (all 3)", "MRR@10": round(mrr, 4), "Avg ms/query": round(t*1000, 3)})
print(f"Intersection MRR: {mrr:.4f}, {t*1000:.1f} ms/q")

Running query_batch on 100 queries...
BM25 batch: 64/100
BM25 batch: 100/100


Batches: 100%|██████████| 1/1 [00:00<00:00, 336.49it/s]

Intersection MRR: 0.1674, 379.6 ms/q


## Results

In [7]:
df = pd.DataFrame(results).sort_values("MRR@10", ascending=False)
df.to_csv("ablation_hybrid_methods.csv", index=False)
print(df.to_string(index=False))

                         Strategy  MRR@10  Avg ms/query
           Dense 0.9 + TF-IDF 0.1  0.5587       377.661
                       Dense only  0.5363       379.779
           Dense 0.75 + BM25 0.25  0.5316       402.249
         Dense 0.75 + TF-IDF 0.25  0.4949       385.011
Dense 0.6 + BM25 0.3 + TF-IDF 0.1  0.4872       387.477
                       RRF (k=30)  0.4007       385.437
                       RRF (k=10)  0.3994       385.088
                       RRF (k=60)  0.3986       388.860
                      RRF (k=100)  0.3975       387.345
           Dense 0.5 + TF-IDF 0.5  0.3899       386.508
         Dense 0.25 + TF-IDF 0.75  0.3135       380.077
                      TF-IDF only  0.2419       385.009
             Intersection (all 3)  0.1674       379.559
